# Next-POI Recommendation: Raw vs Context-Enriched Ranking

**Research task.** Given a user and a chronologically ordered sequence of POI
check-ins, rank candidate locations so that the actual next POI appears as high
as possible.

The notebook evaluates three learned rankers:

1. **LightGBMRanker**
2. **CatBoostRanker**
3. **NeuralTransitionRanker**

The same temporal train/test trail split and candidate-generation procedure are
used for both the raw and enriched versions.

Evaluation uses **MRR**, **Recall@K**, and **NDCG@K** for `K = 5, 10, 20`.

> Metrics are computed on a sampled candidate pool containing the true next POI
> and up to `NEGATIVES_PER_POSITIVE` hard negatives. They are not full-catalogue
> recommendation metrics.

> The enriched configuration passes all available cleaned weather, holiday, opening-status, POI-metadata, and public-transport fields into the model feature matrix. A runtime validation checks that available context columns were not silently omitted.

## 1. Install dependencies (Colab)

In [ ]:
%pip -q install lightgbm catboost pyarrow tqdm

## 2. Mount Google Drive

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Google Drive mounting skipped because this is not a Colab runtime.")

## 3. Imports

In [ ]:
from pathlib import Path
from collections import defaultdict, Counter
import math, random, warnings, json, os, gc, pickle, shutil
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import lightgbm as lgb
from catboost import CatBoostRanker, Pool
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
print("Imports OK | torch", torch.__version__, "| cuda:", torch.cuda.is_available())

## 4. Configuration — paths, columns, hyperparameters

In [ ]:
# ------------------------------------------------------------------
# Project and city configuration
# ------------------------------------------------------------------
DEFAULT_PROJECT_ROOT = (
    "/content/drive/MyDrive/context_trails_data"
    if Path("/content/drive/MyDrive").exists()
    else str(Path.cwd().resolve().parent)
)
PROJECT_ROOT = Path(os.environ.get("POI_PROJECT_ROOT", DEFAULT_PROJECT_ROOT))
DATASET_DIR = PROJECT_ROOT / "dataset_versions"
OUTPUT_ROOT = PROJECT_ROOT / "modeling_results"

CITY_NAME = "PetalingJaya"  # "Tokyo", "PetalingJaya", or "NewYorkCity"

CITY_DATASET_FILES = {
    "Tokyo": {
        "raw": "Tokyo_V1_preprocessed_unsplit_all_rows.parquet",
        "enriched": "Tokyo_V2_preprocessed_unsplit_all_rows.parquet",
    },
    "PetalingJaya": {
        "raw": "PetalingJaya_V1_preprocessed_unsplit_all_rows.parquet",
        "enriched": "PetalingJaya_V2_preprocessed_unsplit_all_rows.parquet",
    },
    "NewYorkCity": {
        "raw": "NewYorkCity_V1_preprocessed_unsplit_all_rows.parquet",
        "enriched": "NewYorkCity_V2_preprocessed_unsplit_all_rows.parquet",
    },
}

if CITY_NAME not in CITY_DATASET_FILES:
    raise ValueError(f"Unsupported city: {CITY_NAME}")

RAW_DATASET_PATH = DATASET_DIR / CITY_DATASET_FILES[CITY_NAME]["raw"]
ENRICHED_DATASET_PATH = DATASET_DIR / CITY_DATASET_FILES[CITY_NAME]["enriched"]

DATASETS = [
    {"city": CITY_NAME, "version": "raw", "path": RAW_DATASET_PATH},
    {"city": CITY_NAME, "version": "enriched", "path": ENRICHED_DATASET_PATH},
]

OUTPUT_DIR = OUTPUT_ROOT / CITY_NAME / "next_poi_3ml_all_context_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = OUTPUT_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CAND_CHUNK_ROWS = 25_000

# Column configuration
CITY_COL = "city"
TRAIL_COL = "trail_id"
USER_COL = "user_id"
VENUE_COL = "venue_id"
TARGET_POI_COL = "target_next_venue_id_clean"
TIME_COL = "timestamp_parsed"
RAW_TIME_COL = "timestamp"
LAT_COL = "latitude_clean"
LON_COL = "longitude_clean"
CATEGORY_COL = "category_clean"
CATEGORY_LVL_COL = "category_lvlFs_clean"
PREV_VENUE_COL = "prev_venue_id_clean"
STEP_COL = "step_index"


# Context available at the current check-in / prediction moment.
QUERY_CONTEXT_FEATURES = {
    "q_temp": ["temp_imputed", "temp"],
    "q_precip": ["precip_capped", "precip_imputed", "precip"],
    "q_windspeed": ["windspeed_imputed", "windspeed"],
    "q_conditions": ["conditions_clean", "conditions"],
    "q_preciptype": ["preciptype_clean", "preciptype"],
    "q_holiday_name": ["holiday_name_clean", "holiday_name"],
    "q_is_public_holiday": [
        "holiday_is_public_holiday_clean",
        "holiday_is_public_holiday",
    ],
    "q_days_since_previous_holiday": [
        "holiday_days_since_previous_imputed",
        "holiday_days_since_previous",
    ],
    "q_days_until_next_holiday": [
        "holiday_days_until_next_imputed",
        "holiday_days_until_next",
    ],
    "q_opening_status": ["opening_status"],
    "q_visit_time_bucket": ["visit_time_bucket_clean", "visit_time_bucket"],
}

QUERY_CONTEXT_FLAGS = [
    "temp_was_imputed",
    "precip_was_imputed",
    "windspeed_was_imputed",
    "conditions_was_missing",
    "preciptype_was_imputed",
    "holiday_name_was_imputed",
    "holiday_is_public_holiday_was_imputed",
    "opening_status_was_missing",
    "visit_time_bucket_was_missing",
]

TRANSIT_COUNT_COLS = [
    f"public_transport_{mode}_count_{radius}m"
    for radius in (250, 500, 1000)
    for mode in ("stop", "bus_stop", "rail_stop", "subway_stop", "metro_stop")
]

# Static metadata attached to each candidate POI.
CANDIDATE_CONTEXT_COLS = [
    "price_imputed",
    "rating_imputed",
    "total_ratings_imputed",
    "total_tips_imputed",
    "log1p_total_ratings_capped",
    "log1p_total_tips_capped",
    "log1p_total_ratings_imputed",
    "log1p_total_tips_imputed",
    "has_poi_metadata",
    "public_transport_nearest_stop_km_capped",
    "public_transport_nearest_stop_km_imputed",
    "public_transport_nearest_stop_feed_clean",
    "public_transport_nearest_stop_mode_clean",
    "public_transport_has_stop_250m_status",
    "public_transport_has_stop_500m_status",
    "public_transport_has_stop_1000m_status",
    "public_transport_nearest_stop_km_was_imputed",
    "public_transport_nearest_stop_feed_was_missing",
    "public_transport_nearest_stop_mode_was_missing",
    "public_transport_has_stop_250m_was_missing",
    "public_transport_has_stop_500m_was_missing",
    "public_transport_has_stop_1000m_was_missing",
]

for col in TRANSIT_COUNT_COLS:
    CANDIDATE_CONTEXT_COLS.extend([
        f"{col}_imputed",
        f"log1p_{col}_imputed",
        f"{col}_was_imputed",
    ])

# Split and evaluation
TRAIN_RATIO = 0.80
EVAL_KS = [5, 10, 20]

# Candidate pool
NEGATIVES_PER_POSITIVE = 10
MAX_CANDIDATES_EVAL = 30
MAX_TRAIN_ROWS = None
MAX_TEST_ROWS = None
SHOW_PROGRESS = True

# Neural transition ranker
NTR_EPOCHS = 10
NTR_BATCH_SIZE = 2_048
NTR_LR = 1e-3
NTR_EMB_DIM = 16
NTR_HIDDEN_DIM = 128
NTR_DROPOUT = 0.2
NTR_WEIGHT_POSITIVES = True
NTR_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("City:", CITY_NAME)
print("Raw dataset:", RAW_DATASET_PATH)
print("Enriched dataset:", ENRICHED_DATASET_PATH)
print("Output:", OUTPUT_DIR)
print("Device:", NTR_DEVICE)

**Run one city at a time.** Change only `CITY_NAME` and, when necessary,
`PROJECT_ROOT`. The raw and enriched filenames are resolved automatically.
Re-run the notebook for each city so that each city receives its own output
and cache directory.

## 5. Loading & preprocessing

In [ ]:
def load_table(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() in [".pkl", ".pickle"]:
        return pd.read_pickle(path)
    raise ValueError(f"Unsupported file type: {path.suffix}")


def ensure_timestamp(df):
    df = df.copy()
    if TIME_COL in df.columns:
        df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    elif RAW_TIME_COL in df.columns:
        df[TIME_COL] = pd.to_datetime(df[RAW_TIME_COL], errors="coerce")
    else:
        raise ValueError(f"Could not find {TIME_COL} or {RAW_TIME_COL}")
    return df


def add_fallback_clean_columns(df):
    # If *_clean / *_imputed feature columns are missing, fall back to raw ones.
    df = df.copy()
    fallback_pairs = [
        (LAT_COL, "latitude"), (LON_COL, "longitude"),
        (CATEGORY_COL, "category"), (CATEGORY_LVL_COL, "category_lvlFs"),
        (PREV_VENUE_COL, "prev_venue_id"),
        ("price_imputed", "price"), ("rating_imputed", "rating"),
        ("total_ratings_imputed", "total_ratings"), ("total_tips_imputed", "total_tips"),
        ("log1p_total_ratings_imputed", "log1p_total_ratings"),
        ("log1p_total_tips_imputed", "log1p_total_tips"),
        ("minutes_since_prev_imputed", "minutes_since_prev"),
    ]
    for clean_col, raw_col in fallback_pairs:
        if clean_col not in df.columns and raw_col in df.columns:
            df[clean_col] = df[raw_col]
    defaults = {
        LAT_COL: np.nan, LON_COL: np.nan, CATEGORY_COL: "unknown",
        CATEGORY_LVL_COL: "unknown", PREV_VENUE_COL: "unknown", STEP_COL: 0,
        "minutes_since_prev": 0, "minutes_since_prev_imputed": 0,
    }
    for c, default in defaults.items():
        if c not in df.columns:
            df[c] = default
    return df


def prepare_df(df):
    df = ensure_timestamp(df)
    df = add_fallback_clean_columns(df)
    df = df.sort_values([TRAIL_COL, TIME_COL, STEP_COL], kind="stable").reset_index(drop=True)
    required = [CITY_COL, TRAIL_COL, USER_COL, VENUE_COL, TARGET_POI_COL, TIME_COL]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    df["hour"]       = df[TIME_COL].dt.hour.fillna(0).astype(int)
    df["weekday"]    = df[TIME_COL].dt.dayofweek.fillna(0).astype(int)
    df["is_weekend"] = df["weekday"].isin([5, 6]).astype(int)
    df = df[df[TARGET_POI_COL].notna()].copy()
    for c in [CITY_COL, TRAIL_COL, USER_COL, VENUE_COL, TARGET_POI_COL,
              PREV_VENUE_COL, CATEGORY_COL, CATEGORY_LVL_COL]:
        if c in df.columns:
            df[c] = df[c].astype(str)
    return df.sort_values([TRAIL_COL, TIME_COL, STEP_COL], kind="stable").reset_index(drop=True)


def print_shape_report(name, df):
    print(f"{name}: rows={len(df):,}, cols={df.shape[1]:,}, "
          f"trails={df[TRAIL_COL].nunique():,}, users={df[USER_COL].nunique():,}, "
          f"venues={df[VENUE_COL].nunique():,}")


def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0088
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return R * 2 * np.arcsin(np.sqrt(a))

## 6. Ranking metrics

In [ ]:
def recall_at_k(y_true, ranked, k):
    return float(str(y_true) in [str(x) for x in ranked[:k]])


def ndcg_at_k(y_true, ranked, k):
    ranked = [str(x) for x in ranked[:k]]
    y_true = str(y_true)
    if y_true not in ranked:
        return 0.0
    return 1.0 / math.log2(ranked.index(y_true) + 2)


def reciprocal_rank(y_true, ranked):
    ranked = [str(x) for x in ranked]
    y_true = str(y_true)
    if y_true not in ranked:
        return 0.0
    return 1.0 / (ranked.index(y_true) + 1)


def evaluate_ranked_lists(y_true_list, ranked_lists, ks=EVAL_KS):
    rows = []
    for y, ranked in zip(y_true_list, ranked_lists):
        row = {"MRR": reciprocal_rank(y, ranked)}
        for k in ks:
            row[f"Recall@{k}"] = recall_at_k(y, ranked, k)
            row[f"NDCG@{k}"]   = ndcg_at_k(y, ranked, k)
        rows.append(row)
    if not rows:
        out = {"MRR": np.nan}
        for k in ks:
            out[f"Recall@{k}"] = np.nan
            out[f"NDCG@{k}"] = np.nan
        return out
    return pd.DataFrame(rows).mean().to_dict()


def print_metric_dict(m):
    return {k: round(float(v), 4) for k, v in m.items()
            if isinstance(v, (int, float, np.integer, np.floating)) and pd.notna(v)}

## 7. Load raw + enriched and build one shared split

In [ ]:
loaded = {}
for item in DATASETS:
    df = prepare_df(load_table(item["path"]))
    loaded[(item["city"], item["version"])] = df
    print_shape_report(f'{item["city"]} / {item["version"]}', df)

city = CITY_NAME
assert (city, "raw") in loaded and (city, "enriched") in loaded


def make_city_split(df_raw, train_ratio=TRAIN_RATIO):
    trail_times = df_raw.groupby(TRAIL_COL)[TIME_COL].min().sort_values()
    trail_ids = trail_times.index.to_numpy()
    n_train = int(len(trail_ids) * train_ratio)
    return set(trail_ids[:n_train]), set(trail_ids[n_train:])


def apply_split(df, train_trails, test_trails):
    return (df[df[TRAIL_COL].isin(train_trails)].copy(),
            df[df[TRAIL_COL].isin(test_trails)].copy())


# The split is derived from the raw dataset and reused for both versions so the
# comparison is fair (identical train/test trails).
train_trails, test_trails = make_city_split(loaded[(city, "raw")])
print(f"\n{city} | train trails: {len(train_trails)} | test trails: {len(test_trails)}")
for version in ["raw", "enriched"]:
    tr, te = apply_split(loaded[(city, version)], train_trails, test_trails)
    print_shape_report(f"{version} train", tr)
    print_shape_report(f"{version} test ", te)

## 8. Training statistics (popularity + Markov transitions + venue metadata)

In [ ]:
def build_stats(train_df):
    stats = {}

    stats["city_pop"] = {
        str(city): group[TARGET_POI_COL]
        .value_counts()
        .index.astype(str)
        .tolist()
        for city, group in train_df.groupby(CITY_COL)
    }
    stats["global_pop"] = (
        train_df[TARGET_POI_COL]
        .value_counts()
        .index.astype(str)
        .tolist()
    )

    transitions = defaultdict(Counter)
    for current, next_poi in zip(
        train_df[VENUE_COL].astype(str),
        train_df[TARGET_POI_COL].astype(str),
    ):
        transitions[current][next_poi] += 1
    stats["transitions"] = transitions

    base_meta_cols = [
        VENUE_COL,
        CITY_COL,
        LAT_COL,
        LON_COL,
        CATEGORY_COL,
        CATEGORY_LVL_COL,
    ]
    meta_cols = list(dict.fromkeys(base_meta_cols + CANDIDATE_CONTEXT_COLS))
    meta_cols = [col for col in meta_cols if col in train_df.columns]

    venue_meta = (
        train_df
        .sort_values(TIME_COL)
        .drop_duplicates(VENUE_COL, keep="last")
        [meta_cols]
        .set_index(VENUE_COL)
    )

    target_counts = train_df[TARGET_POI_COL].value_counts()
    venue_meta["venue_target_popularity"] = (
        venue_meta.index.map(target_counts).fillna(0).astype(float)
    )

    stats["venue_meta"] = venue_meta
    return stats


def top_markov(transitions, cur, n=None):
    return [v for v, _ in transitions.get(str(cur), Counter()).most_common(n)]


def fallback_popularity(stats, city, n=None):
    ranked = stats["city_pop"].get(str(city), []) or stats["global_pop"]
    return ranked if n is None else ranked[:n]


def unique_preserve_order(items):
    out, seen = [], set()
    for item in items:
        item = str(item)
        if item not in seen:
            seen.add(item)
            out.append(item)
    return out


## 9. Candidate pool + feature construction

For each query row we build a small candidate set: the true next POI plus hard
negatives from (a) the current venue's Markov successors and (b) city popularity.
Each candidate becomes one feature row describing the query (current venue, prev
venue, time, category) and the candidate (distance, transition prob, popularity,
and — in `enriched` mode — POI metadata). Every model below consumes this same
representation.

In [ ]:
def get_candidate_pool(row, stats, n_negatives=NEGATIVES_PER_POSITIVE,
                       max_candidates=MAX_CANDIDATES_EVAL, include_true=True):
    true_next = str(row[TARGET_POI_COL]) if include_true and TARGET_POI_COL in row else None
    candidates = []
    if true_next and true_next != "nan":
        candidates.append(true_next)
    candidates += top_markov(stats["transitions"], row[VENUE_COL], n=max_candidates)
    candidates += fallback_popularity(stats, row[CITY_COL], n=max_candidates)
    candidates = unique_preserve_order([c for c in candidates if pd.notna(c)])
    if include_true and true_next and len(candidates) > n_negatives + 1:
        candidates = [true_next] + [c for c in candidates if c != true_next][:n_negatives]
    elif not include_true:
        candidates = candidates[:n_negatives]
    return candidates


def first_available(series, column_names, default=np.nan):
    for col in column_names:
        if col in series.index:
            value = series.get(col)
            if pd.notna(value):
                return value
    return default


def candidate_feature_rows(row, candidates, stats, feature_mode="raw"):
    rows, valid = [], []
    meta = stats["venue_meta"]

    current_poi = str(row[VENUE_COL])
    city = str(row[CITY_COL])
    current_transitions = stats["transitions"].get(current_poi, Counter())
    total_from_current = sum(current_transitions.values())

    current_category = str(row.get(CATEGORY_COL, "unknown"))
    current_category_lvl = str(row.get(CATEGORY_LVL_COL, "unknown"))

    for candidate in candidates:
        candidate = str(candidate)
        if candidate not in meta.index:
            continue

        candidate_meta = meta.loc[candidate]
        candidate_category = str(candidate_meta.get(CATEGORY_COL, "unknown"))
        candidate_category_lvl = str(candidate_meta.get(CATEGORY_LVL_COL, "unknown"))

        coordinates_missing = (
            pd.isna(row.get(LAT_COL, np.nan))
            or pd.isna(row.get(LON_COL, np.nan))
            or pd.isna(candidate_meta.get(LAT_COL, np.nan))
            or pd.isna(candidate_meta.get(LON_COL, np.nan))
        )

        if coordinates_missing:
            distance_km = np.nan
        else:
            distance_km = float(
                haversine_km(
                    row[LAT_COL],
                    row[LON_COL],
                    candidate_meta[LAT_COL],
                    candidate_meta[LON_COL],
                )
            )

        transition_count = current_transitions.get(candidate, 0)
        candidate_popularity = float(
            candidate_meta.get("venue_target_popularity", 0)
        )

        record = {
            "q_city": city,
            "q_user_id": str(row.get(USER_COL, "unknown")),
            "q_current_venue_id": current_poi,
            "q_prev_venue_id": str(row.get(PREV_VENUE_COL, "unknown")),
            "q_category": current_category,
            "q_category_lvl": current_category_lvl,
            "q_hour": int(row.get("hour", 0)),
            "q_weekday": int(row.get("weekday", 0)),
            "q_is_weekend": int(row.get("is_weekend", 0)),
            "q_step_index": float(
                row.get(STEP_COL, 0)
                if pd.notna(row.get(STEP_COL, 0))
                else 0
            ),
            "q_minutes_since_prev": float(
                row.get(
                    "minutes_since_prev_imputed",
                    row.get("minutes_since_prev", 0),
                )
                or 0
            ),
            "cand_venue_id": candidate,
            "cand_city": str(candidate_meta.get(CITY_COL, "unknown")),
            "cand_category": candidate_category,
            "cand_category_lvl": candidate_category_lvl,
            "same_city": int(
                city == str(candidate_meta.get(CITY_COL, "unknown"))
            ),
            "same_category": int(current_category == candidate_category),
            "same_category_lvl": int(
                current_category_lvl == candidate_category_lvl
            ),
            "distance_km": distance_km,
            "log1p_distance_km": (
                np.log1p(distance_km)
                if pd.notna(distance_km)
                else np.nan
            ),
            "transition_count": transition_count,
            "transition_prob": (
                transition_count / total_from_current
                if total_from_current > 0
                else 0.0
            ),
            "candidate_popularity": candidate_popularity,
            "log1p_candidate_popularity": np.log1p(candidate_popularity),
        }

        if feature_mode == "enriched":
            for output_col, source_cols in QUERY_CONTEXT_FEATURES.items():
                record[output_col] = first_available(
                    row,
                    source_cols,
                    default=np.nan,
                )

            for col in QUERY_CONTEXT_FLAGS:
                if col in row.index:
                    record[f"q_{col}"] = row.get(col, np.nan)

            for col in CANDIDATE_CONTEXT_COLS:
                if col in candidate_meta.index:
                    record[f"cand_{col}"] = candidate_meta.get(col, np.nan)

            precipitation = record.get("q_precip", np.nan)
            record["precip_x_distance"] = (
                float(precipitation) * distance_km
                if pd.notna(precipitation) and pd.notna(distance_km)
                else np.nan
            )
            record["weekend_x_candidate_popularity"] = (
                record["q_is_weekend"]
                * record["log1p_candidate_popularity"]
            )

            transit_distance = record.get(
                "cand_public_transport_nearest_stop_km_capped",
                record.get(
                    "cand_public_transport_nearest_stop_km_imputed",
                    np.nan,
                ),
            )
            record["distance_x_transit_distance"] = (
                distance_km * float(transit_distance)
                if pd.notna(distance_km) and pd.notna(transit_distance)
                else np.nan
            )

        rows.append(record)
        valid.append(candidate)

    return pd.DataFrame(rows), valid


def validate_enriched_feature_matrix(X, train_df, stats):
    expected_query_outputs = [
        output_col
        for output_col, source_cols in QUERY_CONTEXT_FEATURES.items()
        if any(source_col in train_df.columns for source_col in source_cols)
    ]
    expected_candidate_outputs = [
        f"cand_{col}"
        for col in CANDIDATE_CONTEXT_COLS
        if col in stats["venue_meta"].columns
    ]

    expected_outputs = expected_query_outputs + expected_candidate_outputs
    missing_outputs = [col for col in expected_outputs if col not in X.columns]

    context_cols = [
        col for col in X.columns
        if any(
            keyword in col
            for keyword in (
                "temp",
                "precip",
                "wind",
                "conditions",
                "holiday",
                "opening",
                "public_transport",
                "rating",
                "price",
                "tips",
            )
        )
    ]

    print(f"Enriched context columns passed to the models: {len(context_cols)}")
    for col in context_cols:
        print(" -", col)

    if missing_outputs:
        raise RuntimeError(
            "Configured context columns were available in the enriched dataset "
            f"but did not reach X: {missing_outputs}"
        )

    weather_available = any(
        source_col in train_df.columns
        for source_cols in QUERY_CONTEXT_FEATURES.values()
        for source_col in source_cols
        if any(
            keyword in source_col
            for keyword in ("temp", "precip", "wind", "conditions")
        )
    )
    transit_available = any(
        col in stats["venue_meta"].columns
        for col in CANDIDATE_CONTEXT_COLS
        if col.startswith("public_transport_")
    )

    if weather_available and not any(
        any(
            keyword in col
            for keyword in ("temp", "precip", "wind", "conditions")
        )
        for col in X.columns
    ):
        raise RuntimeError("Weather columns exist but did not reach the model matrix.")

    if transit_available and not any(
        "public_transport" in col for col in X.columns
    ):
        raise RuntimeError(
            "Public-transport columns exist but did not reach the model matrix."
        )


def build_ranker_dataset(df, stats, feature_mode="raw", max_rows=None,
                         n_negatives=NEGATIVES_PER_POSITIVE):
    if max_rows is not None and len(df) > max_rows:
        df = df.sample(max_rows, random_state=RANDOM_STATE).sort_values(
            [TRAIL_COL, TIME_COL, STEP_COL]
        )

    rows_all, labels, groups, query_ids, y_true = [], [], [], [], []
    iterator = tqdm(
        df.iterrows(),
        total=len(df),
        desc=f"Building candidates ({feature_mode})",
        leave=False,
        disable=not SHOW_PROGRESS,
    )

    for query_id, (_, row) in enumerate(iterator):
        candidates = get_candidate_pool(
            row,
            stats,
            n_negatives=n_negatives,
            include_true=True,
        )
        X_query, valid_candidates = candidate_feature_rows(
            row,
            candidates,
            stats,
            feature_mode,
        )
        if len(X_query) == 0:
            continue

        true_next = str(row[TARGET_POI_COL])
        rows_all.append(X_query)
        labels += [
            1 if candidate == true_next else 0
            for candidate in valid_candidates
        ]
        groups.append(len(X_query))
        query_ids += [query_id] * len(X_query)
        y_true.append(true_next)

    if not rows_all:
        return (
            pd.DataFrame(),
            np.array([], dtype=int),
            [],
            np.array([], dtype=int),
            [],
        )

    return (
        pd.concat(rows_all, ignore_index=True),
        np.array(labels, dtype=int),
        groups,
        np.array(query_ids),
        y_true,
    )


## 9b. Crash-safe cache (resume after a Colab disconnect)

Candidate building is the expensive step, so it is checkpointed to Drive every
`CAND_CHUNK_ROWS` source rows. If the runtime dies mid-build (e.g. at 69%),
re-running the run cell **skips every chunk already written** and continues from
where it stopped — you lose at most the one in-progress chunk. Once a full
`{version}/{split}` candidate set is assembled it is cached as a single parquet and
reloaded instantly on any later run. Finished `(version, model)` results are also
saved immediately and skipped on rerun.

To force a clean rebuild, delete the `cache/` folder (or `next_poi_results.csv`)
inside the output directory.

In [ ]:
def _cand_final_paths(version, split):
    return (CACHE_DIR / f"cand_{version}_{split}.parquet",
            CACHE_DIR / f"cand_{version}_{split}.meta.pkl")


def build_candidates_resumable(df, stats, feature_mode, version, split,
                               n_negatives, max_rows=None, chunk_rows=CAND_CHUNK_ROWS):
    final_px, final_pk = _cand_final_paths(version, split)
    if final_px.exists() and final_pk.exists():
        X = pd.read_parquet(final_px)
        meta = pickle.load(open(final_pk, "rb"))
        print(f"  loaded cached candidates: {version}/{split}  rows={len(X):,} queries={len(meta['groups']):,}")
        return X, meta["y"], meta["groups"], meta["q"], meta["ytrue"]

    if max_rows is not None and len(df) > max_rows:
        df = df.sample(max_rows, random_state=RANDOM_STATE)
    df = df.sort_values([TRAIL_COL, TIME_COL, STEP_COL]).reset_index(drop=True)

    part_dir = CACHE_DIR / f"cand_{version}_{split}_parts"
    part_dir.mkdir(parents=True, exist_ok=True)
    n_chunks = max(1, math.ceil(len(df) / chunk_rows))

    for ci in range(n_chunks):
        ppx = part_dir / f"part_{ci:05d}.parquet"
        ppk = part_dir / f"part_{ci:05d}.meta.pkl"
        if ppx.exists() and ppk.exists():
            continue  # chunk finished in an earlier (crashed) run
        sub = df.iloc[ci * chunk_rows:(ci + 1) * chunk_rows]
        rows_all, labels, groups, ytrue = [], [], [], []
        for _, row in tqdm(sub.iterrows(), total=len(sub),
                           desc=f"{version}/{split} chunk {ci+1}/{n_chunks}",
                           leave=False, disable=not SHOW_PROGRESS):
            cands = get_candidate_pool(row, stats, n_negatives=n_negatives, include_true=True)
            Xq, valid = candidate_feature_rows(row, cands, stats, feature_mode)
            if len(Xq) == 0:
                continue
            true_next = str(row[TARGET_POI_COL])
            rows_all.append(Xq)
            labels += [1 if c == true_next else 0 for c in valid]
            groups.append(len(Xq))
            ytrue.append(true_next)
        Xc = pd.concat(rows_all, ignore_index=True) if rows_all else pd.DataFrame()
        Xc.to_parquet(ppx, index=False)
        pickle.dump({"y": labels, "groups": groups, "ytrue": ytrue}, open(ppk, "wb"))

    # Assemble all chunks, assigning global query ids.
    Xs, y, groups, q, ytrue, qid = [], [], [], [], [], 0
    for ci in range(n_chunks):
        Xc = pd.read_parquet(part_dir / f"part_{ci:05d}.parquet")
        meta = pickle.load(open(part_dir / f"part_{ci:05d}.meta.pkl", "rb"))
        if not meta["groups"]:
            continue
        Xs.append(Xc)
        y += list(meta["y"])
        ytrue += list(meta["ytrue"])
        for gsize in meta["groups"]:
            groups.append(gsize)
            q += [qid] * gsize
            qid += 1
    X = pd.concat(Xs, ignore_index=True) if Xs else pd.DataFrame()
    y = np.array(y, dtype=int)
    q = np.array(q)
    X.to_parquet(final_px, index=False)
    pickle.dump({"y": y, "groups": groups, "q": q, "ytrue": ytrue}, open(final_pk, "wb"))
    shutil.rmtree(part_dir, ignore_errors=True)   # drop per-chunk parts once assembled
    print(f"  built + cached candidates: {version}/{split}  rows={len(X):,} queries={len(groups):,}")
    return X, y, groups, q, ytrue


RESULTS_CSV = OUTPUT_DIR / "next_poi_results.csv"


def load_prior_results():
    if RESULTS_CSV.exists():
        recs = pd.read_csv(RESULTS_CSV).to_dict("records")
        done = {(str(r["version"]), str(r["model"])) for r in recs}
        return recs, done
    return [], set()


def save_results(results):
    order = ["city", "version", "model", "MRR"] + \
            [f"Recall@{k}" for k in EVAL_KS] + [f"NDCG@{k}" for k in EVAL_KS]
    rdf = pd.DataFrame(results)
    rdf = rdf[[c for c in order if c in rdf.columns]].round(4)
    rdf.to_csv(RESULTS_CSV, index=False)
    return rdf

## 10. Models 1 & 2 — LightGBM / CatBoost learning-to-rank

In [ ]:
def prepare_lgb_features(X_train, X_test):
    X_train, X_test = X_train.copy(), X_test.copy()
    cat_cols = []
    for c in X_train.columns:
        if X_train[c].dtype == "object":
            cat_cols.append(c)
            cats = pd.Categorical(X_train[c].astype(str).fillna("missing")).categories
            X_train[c] = pd.Categorical(X_train[c].astype(str).fillna("missing"), categories=cats)
            X_test[c]  = pd.Categorical(X_test[c].astype(str).fillna("missing"),  categories=cats)
        else:
            X_train[c] = pd.to_numeric(X_train[c], errors="coerce")
            X_test[c]  = pd.to_numeric(X_test[c],  errors="coerce")
    return X_train, X_test, cat_cols


def prepare_cat_features(X_train, X_test):
    X_train, X_test = X_train.copy(), X_test.copy()
    cat_cols = []
    for c in X_train.columns:
        if X_train[c].dtype == "object":
            cat_cols.append(c)
            X_train[c] = X_train[c].astype(str).fillna("missing")
            X_test[c]  = X_test[c].astype(str).fillna("missing")
        else:
            X_train[c] = pd.to_numeric(X_train[c], errors="coerce").fillna(-999)
            X_test[c]  = pd.to_numeric(X_test[c],  errors="coerce").fillna(-999)
    return X_train, X_test, [X_train.columns.get_loc(c) for c in cat_cols]


def train_ranker(ranker_type, X_train, y_train, groups_train, q_train, X_test):
    if ranker_type.lower() == "lightgbm":
        Xtr, Xte, cat_cols = prepare_lgb_features(X_train, X_test)
        model = lgb.LGBMRanker(objective="lambdarank", metric="ndcg", n_estimators=300,
                               learning_rate=0.05, num_leaves=63, subsample=0.9,
                               colsample_bytree=0.9, random_state=RANDOM_STATE, n_jobs=-1)
        model.fit(Xtr, y_train, group=groups_train,
                  categorical_feature=cat_cols if cat_cols else "auto")
        return {"model": model, "type": "lightgbm"}, Xtr, Xte
    Xtr, Xte, cat_idx = prepare_cat_features(X_train, X_test)
    pool = Pool(Xtr, y_train, group_id=q_train, cat_features=cat_idx)
    model = CatBoostRanker(iterations=300, learning_rate=0.05, depth=6,
                           loss_function="YetiRank", eval_metric="NDCG",
                           random_seed=RANDOM_STATE, verbose=100)
    model.fit(pool)
    return {"model": model, "type": "catboost"}, Xtr, Xte


def ranked_lists_from_ranker(bundle, X_test_prepared, query_ids, y_true):
    scores = bundle["model"].predict(X_test_prepared)
    pred_df = pd.DataFrame({"query_id": query_ids,
                            "candidate": X_test_prepared["cand_venue_id"].astype(str).to_numpy(),
                            "score": scores})
    ranked_lists = []
    for qid in sorted(pred_df["query_id"].unique()):
        ranked_lists.append(pred_df[pred_df["query_id"] == qid]
                            .sort_values("score", ascending=False)["candidate"].tolist())
    return y_true, ranked_lists

## 11. Model 3 — Neural Context-Aware Transition Ranker

A compact PyTorch MLP: each categorical feature gets its own embedding, numeric
features are standardised, and a 2-hidden-layer network scores each candidate.
Trained with class-weighted BCE (1 positive vs many negatives per query).

In [ ]:
class NTRPreprocessor:
    def __init__(self):
        self.cat_cols, self.num_cols, self.cat_maps = [], [], {}
        self.num_mean = self.num_std = None

    def fit(self, X):
        X = X.copy()
        self.cat_cols = [c for c in X.columns
                         if X[c].dtype == "object" or str(X[c].dtype).startswith("category")]
        self.num_cols = [c for c in X.columns if c not in self.cat_cols]
        self.cat_maps = {}
        for c in self.cat_cols:
            vals = pd.Series(X[c]).astype(str).fillna("missing")
            uniques = vals.value_counts().index.tolist()
            self.cat_maps[c] = {v: i + 1 for i, v in enumerate(uniques)}  # 0 reserved for unknown
        if self.num_cols:
            arr = X[self.num_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
            self.num_mean = np.nan_to_num(np.nanmean(arr, axis=0), nan=0.0).astype(np.float32)
            self.num_std  = np.nan_to_num(np.nanstd(arr, axis=0),  nan=1.0).astype(np.float32)
            self.num_std[self.num_std == 0] = 1.0
        else:
            self.num_mean = np.array([], dtype=np.float32)
            self.num_std  = np.array([], dtype=np.float32)
        return self

    def transform(self, X):
        X = X.copy()
        cat_arrays = []
        for c in self.cat_cols:
            vals = pd.Series(X[c]).astype(str).fillna("missing") if c in X.columns \
                   else pd.Series(["missing"] * len(X))
            cat_arrays.append(vals.map(self.cat_maps[c]).fillna(0).to_numpy(dtype=np.int64))
        X_cat = np.vstack(cat_arrays).T.astype(np.int64) if cat_arrays else np.zeros((len(X), 0), dtype=np.int64)
        if self.num_cols:
            for c in self.num_cols:
                if c not in X.columns:
                    X[c] = np.nan
            arr = X[self.num_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
            X_num = np.nan_to_num((arr - self.num_mean) / self.num_std, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        else:
            X_num = np.zeros((len(X), 0), dtype=np.float32)
        return X_cat, X_num

    @property
    def cat_cardinalities(self):
        return [len(self.cat_maps[c]) + 1 for c in self.cat_cols]


class NTRDataset(Dataset):
    def __init__(self, X_cat, X_num, y=None):
        self.X_cat = torch.from_numpy(X_cat).long()
        self.X_num = torch.from_numpy(X_num).float()
        self.y = None if y is None else torch.from_numpy(np.asarray(y, dtype=np.float32)).float()

    def __len__(self):
        return self.X_cat.shape[0]

    def __getitem__(self, i):
        if self.y is None:
            return self.X_cat[i], self.X_num[i]
        return self.X_cat[i], self.X_num[i], self.y[i]


class NeuralTransitionRanker(nn.Module):
    def __init__(self, cat_cardinalities, n_num, emb_dim=16, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.embeddings = nn.ModuleList()
        emb_total = 0
        for card in cat_cardinalities:
            dim = min(emb_dim, max(4, int(round(math.sqrt(max(card, 2))))))
            self.embeddings.append(nn.Embedding(card, dim))
            emb_total += dim
        self.mlp = nn.Sequential(
            nn.Linear(emb_total + n_num, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x_cat, x_num):
        parts = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        if x_num.shape[1] > 0:
            parts.append(x_num)
        return self.mlp(torch.cat(parts, dim=1)).squeeze(-1)


def train_neural_transition_ranker(X_train, y_train, X_test=None):
    pre = NTRPreprocessor().fit(X_train)
    Xtr_cat, Xtr_num = pre.transform(X_train)
    train_ds = NTRDataset(Xtr_cat, Xtr_num, y_train)
    model = NeuralTransitionRanker(pre.cat_cardinalities, n_num=Xtr_num.shape[1],
                                   emb_dim=NTR_EMB_DIM, hidden_dim=NTR_HIDDEN_DIM,
                                   dropout=NTR_DROPOUT).to(NTR_DEVICE)
    if NTR_WEIGHT_POSITIVES:
        n_pos = max(1, float(np.sum(y_train == 1)))
        n_neg = max(1, float(np.sum(y_train == 0)))
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([n_neg / n_pos], device=NTR_DEVICE))
    else:
        loss_fn = nn.BCEWithLogitsLoss()
    opt = torch.optim.AdamW(model.parameters(), lr=NTR_LR, weight_decay=1e-4)
    loader = DataLoader(train_ds, batch_size=NTR_BATCH_SIZE, shuffle=True,
                        num_workers=0, pin_memory=(NTR_DEVICE == "cuda"))
    for epoch in range(1, NTR_EPOCHS + 1):
        model.train(); losses = []
        for xb_cat, xb_num, yb in tqdm(loader, desc=f"NTR epoch {epoch}/{NTR_EPOCHS}",
                                       leave=False, disable=not SHOW_PROGRESS):
            xb_cat, xb_num, yb = xb_cat.to(NTR_DEVICE), xb_num.to(NTR_DEVICE), yb.to(NTR_DEVICE)
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(xb_cat, xb_num), yb)
            loss.backward(); opt.step()
            losses.append(float(loss.item()))
        print(f"Epoch {epoch}/{NTR_EPOCHS} | loss={np.mean(losses):.4f}")
    return {"model": model, "preprocessor": pre, "type": "neural_transition"}


def neural_transition_predict_scores(bundle, X, batch_size=None):
    batch_size = batch_size or max(1024, int(NTR_BATCH_SIZE))
    model, pre = bundle["model"], bundle["preprocessor"]
    X_cat, X_num = pre.transform(X)
    loader = DataLoader(NTRDataset(X_cat, X_num, y=None), batch_size=batch_size, shuffle=False)
    model.eval(); scores = []
    with torch.no_grad():
        for xb_cat, xb_num in tqdm(loader, desc="NTR scoring", leave=False, disable=not SHOW_PROGRESS):
            s = model(xb_cat.to(NTR_DEVICE), xb_num.to(NTR_DEVICE)).detach().cpu().numpy()
            scores.append(s)
    return np.concatenate(scores) if scores else np.array([], dtype=np.float32)


def ranked_lists_from_neural_transition(bundle, X_test, query_ids, y_true):
    scores = neural_transition_predict_scores(bundle, X_test)
    pred_df = pd.DataFrame({"query_id": query_ids,
                            "candidate": X_test["cand_venue_id"].astype(str).to_numpy(),
                            "score": scores})
    ranked_lists = []
    for qid in sorted(pred_df["query_id"].unique()):
        ranked_lists.append(pred_df[pred_df["query_id"] == qid]
                            .sort_values("score", ascending=False)["candidate"].tolist())
    return y_true, ranked_lists

## 12. Run all 3 models on raw and enriched (resumable)

Candidate datasets are built once per version (checkpointed to Drive) and shared by
all three models. Finished `(version, model)` pairs are skipped, so re-running this
cell after a disconnect continues instead of starting over.

In [ ]:
results, done = load_prior_results()
if done:
    print("Resuming — already completed:", sorted(done))

GBDT = [("lightgbm", "LightGBMRanker"), ("catboost", "CatBoostRanker")]
ALL_MODELS = [m for _, m in GBDT] + ["NeuralTransitionRanker"]

for version in ["raw", "enriched"]:
    pending = [m for m in ALL_MODELS if (version, m) not in done]
    if not pending:
        print(f"\n[{version}] all models already done — skipping.")
        continue

    print("\n" + "=" * 80)
    print("VERSION:", version, "| pending:", pending)
    print("=" * 80)

    df = loaded[(city, version)]
    train_df, test_df = apply_split(df, train_trails, test_trails)
    train_df = train_df[train_df[TARGET_POI_COL].notna()].copy()
    test_df  = test_df[test_df[TARGET_POI_COL].notna()].copy()

    stats = build_stats(train_df)
    feature_mode = "enriched" if version == "enriched" else "raw"

    # Shared candidate datasets (train + test), checkpointed + cached to Drive.
    Xtr, ytr, groups_tr, qtr, ytrue_tr = build_candidates_resumable(
        train_df, stats, feature_mode, version, "train",
        n_negatives=NEGATIVES_PER_POSITIVE, max_rows=MAX_TRAIN_ROWS)
    Xte, yte, groups_te, qte, ytrue_te = build_candidates_resumable(
        test_df, stats, feature_mode, version, "test",
        n_negatives=min(MAX_CANDIDATES_EVAL - 1, NEGATIVES_PER_POSITIVE), max_rows=MAX_TEST_ROWS)
    print(f"Candidate rows: train={len(Xtr):,}, test={len(Xte):,} | "
          f"queries: train={len(groups_tr):,}, test={len(groups_te):,}")

    if len(Xtr) == 0 or len(Xte) == 0:
        print("Empty candidate dataset — skipping version.")
        continue

    if version == "enriched":
        validate_enriched_feature_matrix(Xtr, train_df, stats)

    # ---- Models 1 & 2: GBDT rankers ----
    for ranker_type, model_name in GBDT:
        if (version, model_name) in done:
            continue
        print(f"\n[{version}] Training {model_name} ...")
        bundle, _, Xte_p = train_ranker(ranker_type, Xtr, ytr, groups_tr, qtr, Xte)
        y_eval, ranked = ranked_lists_from_ranker(bundle, Xte_p, qte, ytrue_te)
        m = evaluate_ranked_lists(y_eval, ranked)
        m.update({"city": city, "version": version, "model": model_name})
        results.append(m); done.add((version, model_name))
        save_results(results)   # checkpoint after every model
        print(model_name, print_metric_dict(m))

    # ---- Model 3: Neural transition ranker ----
    if (version, "NeuralTransitionRanker") not in done:
        print(f"\n[{version}] Training NeuralTransitionRanker ...")
        ntr_bundle = train_neural_transition_ranker(Xtr, ytr, Xte)
        y_eval, ranked = ranked_lists_from_neural_transition(ntr_bundle, Xte, qte, ytrue_te)
        m = evaluate_ranked_lists(y_eval, ranked)
        m.update({"city": city, "version": version, "model": "NeuralTransitionRanker"})
        results.append(m); done.add((version, "NeuralTransitionRanker"))
        save_results(results)
        print("NeuralTransitionRanker", print_metric_dict(m))

    del df, train_df, test_df, stats, Xtr, Xte
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_df = save_results(results)
print("\nSaved:", RESULTS_CSV)
results_df

## 13. Raw vs enriched comparison

In [ ]:
def compare_raw_enriched(results_df, metric):
    piv = results_df.pivot_table(index="model", columns="version", values=metric)
    if "enriched" in piv.columns and "raw" in piv.columns:
        piv["delta (enriched - raw)"] = piv["enriched"] - piv["raw"]
    return piv.round(4)

for metric in ["MRR"] + [f"Recall@{k}" for k in EVAL_KS] + [f"NDCG@{k}" for k in EVAL_KS]:
    print("\n" + metric)
    display(compare_raw_enriched(results_df, metric))

## 14. Plots

In [ ]:
plot_metrics = ["MRR", "Recall@5", "Recall@10", "NDCG@10"]
models = results_df["model"].unique().tolist()
versions = ["raw", "enriched"]

fig, axes = plt.subplots(1, len(plot_metrics), figsize=(5 * len(plot_metrics), 4.5))
x = np.arange(len(models))
width = 0.38
for ax, metric in zip(axes, plot_metrics):
    for i, ver in enumerate(versions):
        vals = [results_df[(results_df.model == mdl) & (results_df.version == ver)][metric].mean()
                for mdl in models]
        ax.bar(x + (i - 0.5) * width, vals, width, label=ver)
    ax.set_title(metric)
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=30, ha="right")
    ax.grid(axis="y", alpha=0.3)
    ax.legend()
fig.suptitle(f"{CITY_NAME}: next-POI ranking — raw vs enriched", y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "next_poi_raw_vs_enriched.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved plot:", OUTPUT_DIR / "next_poi_raw_vs_enriched.png")